# Translators Amalgamated Greek NT (TAGNT) Parser

## What is TAGNT?
The **Translators Amalgamated Greek New Testament (TAGNT)** is a comprehensive database created by STEPBible (based on work at Tyndale House Cambridge). It brings together all the Greek words found in major standard editions of the New Testament (like Nestle-Aland, Textus Receptus, SBL, etc.).

For each Greek word in the New Testament, TAGNT provides:
*   **English Translation**: How it is commonly translated.
*   **Dictionary Form (Lemma)**: The root form of the word as you'd find it in a dictionary.
*   **Strong's Numbers**: Standardized numbers assigned to each Greek root word, allowing you to identify the original word even if you don't read Greek.
*   **Grammar/Morphology**: Details about how the word is used in the sentence (e.g., noun, verb, case, tense).
*   **Manuscript Variants**: Information on differences across ancient manuscripts.

## Purpose of this Demonstration
For **non-specialists**, exploring the original Greek text can seem daunting. This notebook bridges that gap. 

It demonstrates how to take the raw, complex TAGNT text files and convert them into a clean, easy-to-read table (a "DataFrame" using Python's `pandas` library). Once the data is in this tabular format, anyone can easily search for a specific verse (like John 1:1) and immediately see the original Greek words side-by-side with their dictionary forms and grammatical roles—no advanced linguistic or programming knowledge required!

---
### Loading the Data
The data is distributed across multiple tab-separated text files. Since the format has changed from earlier simple CSV-like structures, we use `pandas` to parse the amalgamation format, skip the metadata headers, and extract the relevant columns (`Reference`, `GreekWord`, `Lemma`, `Strongs`, and `Morph`).


In [1]:
import pandas as pd
import glob
import re

# We will read the Amalgamated NT files and parse out the necessary columns
files = glob.glob('STEPBible-Data/Translators Amalgamated OT+NT/TAGNT*.txt')

df_list = []
for f in files:
    # Skip metadata lines starting with '#' or '=' or empty lines. 
    # Let pandas auto-detect columns by passing header=None without 'names'
    # This prevents valid data lines with extra tabs from being dropped by on_bad_lines
    df = pd.read_csv(f, sep='\t', encoding='utf-8', on_bad_lines='skip', skiprows=50, low_memory=False, header=None)
    df_list.append(df)

raw_df = pd.concat(df_list, ignore_index=True)

# Filter out lines that do not start with a reference (like 'Mat.1.1#01')
raw_df = raw_df[raw_df[0].astype(str).str.match(r'^[1-3]?\w{2,3}\.\d+\.\d+#')]

# Define raw column names for non-specialist readability
raw_columns = [
    'Raw Reference & Type', 'Greek with Pronunciation', 'English Translation', 
    'Strongs & Grammar', 'Lemma & Gloss', 'Editions', 'Meaning Variants', 
    'Spelling Variants', 'Spanish Translation', 'Sub-meaning', 
    'Conjoin Word', 'Simple Strongs', 'Alternate Strongs', 'Extra'
]
# Assign column names (cutting off any extra unexpected columns that pandas detected)
raw_df.columns = raw_columns[:len(raw_df.columns)] + [f"Extra_{i}" for i in range(len(raw_df.columns) - len(raw_columns))]

raw_df.head()

,Raw Reference & Type,Greek with Pronunciation,English Translation,Strongs & Grammar,Lemma & Gloss,Editions,Meaning Variants,Spelling Variants,Spanish Translation,Sub-meaning,Conjoin Word,Simple Strongs,Alternate Strongs,Extra,Extra_0,Extra_1,Extra_2
44,Mat.1.1#01=NKO,Βίβλος (Biblos),[The] book,G0976=N-NSF,βίβλος=book,NA28+NA27+Tyn+SBL+WH+Treg+TR+Byz,NaN,NaN,Libro,book,#01,G0976,NaN,NaN,NaN,NaN,NaN
45,Mat.1.1#02=NKO,γενέσεως (geneseōs),of [the] genealogy,G1078=N-GSF,γένεσις=origin,NA28+NA27+Tyn+SBL+WH+Treg+TR+Byz,NaN,NaN,de origen,origin,#02,G1078,NaN,NaN,NaN,NaN,NaN
46,Mat.1.1#03=NKO,Ἰησοῦ (Iēsou),of Jesus,G2424G=N-GSM-P,Ἰησοῦς=Jesus/Joshua,NA28+NA27+Tyn+SBL+WH+Treg+TR+Byz,NaN,NaN,de Jesús,Jesus»Jesus|Jesus@Mat.1.1,#03,G2424,NaN,NaN,NaN,NaN,NaN
47,Mat.1.1#04=NKO,Χριστοῦ (Christou),Christ,G5547=N-GSM-T,Χριστός=Christ,NA28+NA27+Tyn+SBL+WH+Treg+TR+Byz,NaN,NaN,Ungido,Christ»Christ|Jesus@Mat.1.1,#04,G5547,NaN,NaN,NaN,NaN,NaN
48,Mat.1.1#05=NKO,υἱοῦ (huiou),son,G5207=N-GSM,υἱός=son,NA28+NA27+Tyn+SBL+WH+Treg+TR+Byz,NaN,NaN,hijo,son,#05,G5207_A,NaN,NaN,NaN,NaN,NaN


### Column Extraction
The raw text format uses an amalgamation pattern where specific values are grouped. For example:
- **Reference & Sequence**: `Mat.1.1#01=NKO`
- **Greek Word**: `Βίβλος (Biblos)`
- **Strongs & Morph**: `G0976=N-NSF`
- **Lemma & Gloss**: `βίβλος=book`

We can use string manipulation (splitting by characters like `#`, `(`, and `=`) to build a clean DataFrame.


In [2]:
# First, load the Morphology dictionary to expand the grammatical codes
morph_file = 'STEPBible-Data/Morphology codes/TEGMC - Translators Expansion of Greek Morphhology Codes - STEPBible.org CC BY.txt'

# Read the file and parse lines mapping code -> expansion
morph_dict = {}
with open(morph_file, 'r', encoding='utf-8') as f:
    for line in f:
        # Stop at the Hebrew morphology section if it exists, but typically this is just Greek
        if '\t' in line and '=' in line and not line.startswith('#'):
            parts = line.strip().split('\t')
            if len(parts) >= 2:
                code = parts[0].strip()
                expansion = parts[1].strip()
                morph_dict[code] = expansion

# Extract the relevant columns into a clean DataFrame with user-friendly names
df_nt = pd.DataFrame({
    'Bible Reference': raw_df['Raw Reference & Type'].str.split('#').str[0],
    'Greek Text': raw_df['Greek with Pronunciation'].str.split(r' \(').str[0].str.strip(),
    'Dictionary Form (Lemma)': raw_df['Lemma & Gloss'].str.split('=').str[0].str.strip(),
    "Strong's Number": raw_df['Strongs & Grammar'].str.split('=').str[0].str.strip(),
    'Grammar Code': raw_df['Strongs & Grammar'].str.split('=').str[1].str.strip()
})

# Map the explicit labels for non-specialists using the dictionary
# If a code isn't found, keep the original code
df_nt['Grammar / Morphology'] = df_nt['Grammar Code'].map(morph_dict).fillna(df_nt['Grammar Code'])

# Drop the raw code column to keep it clean for non-specialists
df_nt = df_nt.drop(columns=['Grammar Code'])

df_nt.head()

,Bible Reference,Greek Text,Dictionary Form (Lemma),Strong's Number,Grammar / Morphology
44,Mat.1.1,Βίβλος,βίβλος,G0976,Function=Noun; Case=Nominative; Number=Singula...
45,Mat.1.1,γενέσεως,γένεσις,G1078,Function=Noun; Case=Genitive; Number=Singular;...
46,Mat.1.1,Ἰησοῦ,Ἰησοῦς,G2424G,Function=Noun; Case=Genitive; Number=Singular;...
47,Mat.1.1,Χριστοῦ,Χριστός,G5547,Function=Noun; Case=Genitive; Number=Singular;...
48,Mat.1.1,υἱοῦ,υἱός,G5207,Function=Noun; Case=Genitive; Number=Singular;...


### Example Query
Now that the data is structured, we can filter for specific verses. Below is an example of extracting the Greek text for **John 1:1** (`Jhn.1.1`).


In [3]:
# Filter for John 1:1
john_1_1 = df_nt[df_nt['Bible Reference'] == 'Jhn.1.1']

# Display the verse data with friendly column names
john_1_1[['Greek Text', "Strong's Number", 'Grammar / Morphology']]

,Greek Text,Strong's Number,Grammar / Morphology
85276,Ἐν,G1722,Function=Preposition
85277,ἀρχῇ,G0746,Function=Noun; Case=Dative; Number=Singular; G...
85278,ἦν,G1510,Function=Verb; Tense=Imperfect; Voice=Active; ...
85279,ὁ,G3588,Function=Definite article; Case=Nominative; Nu...
85280,"λόγος,",G3056,Function=Noun; Case=Nominative; Number=Singula...
85281,καὶ,G2532,Function=Conjunction
85282,ὁ,G3588,Function=Definite article; Case=Nominative; Nu...
85283,λόγος,G3056,Function=Noun; Case=Nominative; Number=Singula...
85284,ἦν,G1510,Function=Verb; Tense=Imperfect; Voice=Active; ...
85285,πρὸς,G4314,Function=Preposition
